### Add small object use YOLO

In [1]:
from ultralytics import YOLO
from pathlib import Path
import json
import numpy as np
import os
from tqdm import tqdm

os.chdir("../../")

model = YOLO("weights/yolo-seg/yolo11s_giftbox.pt")

# YOLO class mapping: {0: 'red box', 1: 'small object', 2: 'green small box', 3: 'toy car'}
SMALL_OBJECT_YOLO_CLS = 1
SMALL_OBJECT_COCO_ID = 1

images_dir = Path("datasets/standard_datasets/images")
anno_dir = Path("datasets/standard_datasets/annotations")

video_ids = [d.name for d in images_dir.iterdir() if d.is_dir()]

total_added = 0

for video_id in tqdm(video_ids, desc="appending small object"):

    # if video_id in ["0001","0002","0003","0004","0005","0006","0012"]:
    #     continue

    image_folder = images_dir / video_id
    annotation_file = anno_dir / f"annotations_{video_id}.json"

    predict_results = model(image_folder, verbose=False)

    with open(annotation_file, "r") as f:
        coco_data = json.load(f)

    file_to_image_id = {img["file_name"]: img["id"] for img in coco_data["images"]}
    annotations = coco_data["annotations"]

    for result in predict_results:
        file_name = Path(result.path).name
        image_id = file_to_image_id.get(file_name)
        if image_id is None:
            continue

        if result.masks is None:
            continue

        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        for i, mask_polygon in enumerate(result.masks.xy):
            if class_ids[i] != SMALL_OBJECT_YOLO_CLS:
                continue

            x1, y1, x2, y2 = boxes_xyxy[i]
            w, h = x2 - x1, y2 - y1

            poly = mask_polygon
            if len(poly) < 3:
                continue
            x = poly[:, 0]
            y = poly[:, 1]
            area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

            annotations.append({
                "id": 0,  # placeholder, re-indexed below
                "image_id": image_id,
                "category_id": SMALL_OBJECT_COCO_ID,
                "segmentation": [mask_polygon.flatten().tolist()],
                "bbox": [float(x1), float(y1), float(w), float(h)],
                "area": float(area),
                "iscrowd": 0
            })
            total_added += 1

    # re-index all annotation IDs sequentially
    for new_id, ann in enumerate(annotations, start=1):
        ann["id"] = new_id

    coco_data["annotations"] = annotations

    with open(annotation_file, "w") as f:
        json.dump(coco_data, f, indent=2)

print(f"Total 'small object' annotations added from YOLO: {total_added}")


appending small object: 100%|██████████| 124/124 [01:21<00:00,  1.52it/s]

Total 'small object' annotations added from YOLO: 9236


In [3]:
# Single folder mode — change VIDEO_ID to target
VIDEO_ID = "0012"

image_folder = images_dir / VIDEO_ID
annotation_file = anno_dir / f"annotations_{VIDEO_ID}.json"

predict_results = model(image_folder, verbose=False)

with open(annotation_file, "r") as f:
    coco_data = json.load(f)

file_to_image_id = {img["file_name"]: img["id"] for img in coco_data["images"]}
annotations = coco_data["annotations"]

added = 0
for result in predict_results:
    file_name = Path(result.path).name
    image_id = file_to_image_id.get(file_name)
    if image_id is None:
        continue

    if result.masks is None:
        continue

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    for i, mask_polygon in enumerate(result.masks.xy):
        if class_ids[i] != SMALL_OBJECT_YOLO_CLS:
            continue

        x1, y1, x2, y2 = boxes_xyxy[i]
        w, h = x2 - x1, y2 - y1

        poly = mask_polygon
        if len(poly) < 3:
            continue
        x = poly[:, 0]
        y = poly[:, 1]
        area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

        annotations.append({
            "id": 0,
            "image_id": image_id,
            "category_id": SMALL_OBJECT_COCO_ID,
            "segmentation": [mask_polygon.flatten().tolist()],
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "area": float(area),
            "iscrowd": 0
        })
        added += 1

for new_id, ann in enumerate(annotations, start=1):
    ann["id"] = new_id

coco_data["annotations"] = annotations

with open(annotation_file, "w") as f:
    json.dump(coco_data, f, indent=2)

print(f"[{VIDEO_ID}] added {added} 'small object' annotations")


[0012] added 87 'small object' annotations
